# Checkpoint2: Research Questions and Method Planning
## Amazon Electronics Ratings Dataset

This notebook defines the research questions, feasibility checks, and methodological plan for the final project.

The goal of this checkpoint is to:
1. briefly recap the dataset and Phase 1 EDA findings,
2. perform additional EDA needed to make the research questions realistic,
3. propose 3 research questions,
4. map each research question to a data mining task, algorithm, and evaluation criteria,
5. run small initial tests to verify that the proposed methods are feasible.

Throughout the notebook, every algorithmic decision is documented and justified.

## 1. Project Scope Recap

### Dataset
The project uses the Amazon Electronics ratings dataset, where each row represents a user-product interaction with:
- `userId`
- `productId`
- `rating`
- `timestamp`

### Relevant characteristics from Phase 1 EDA
Phase 1 showed several important properties of the dataset:

- The user-product matrix is extremely sparse.
- Ratings are heavily concentrated at 4 and 5 stars.
- User activity is long-tailed: most users interact with very few products.
- Product popularity is also long-tailed: a small number of products dominate interaction counts.
- User interactions show temporal burstiness, meaning that ordering may contain useful structure.

### Why this matters
These properties suggest that:
- transaction-style mining is possible, but requires filtering and careful support thresholds,
- popularity bias is likely to dominate naive association rule mining,
- sequential information may reveal patterns that unordered itemsets miss.

In [ ]:

import os
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
!pip install prefixspan

In [ ]:
# ================================
# Imports and configuration
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount("/content/drive")
from collections import Counter
from itertools import islice

# Course techniques
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Optional external technique
# Install if needed in Colab:
#!pip install prefixspan
from prefixspan import PrefixSpan

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# ================================
# Load dataset
# ================================
data_path = "/content/drive/MyDrive/TAMU/CSCE676/data/ratings_Electronics (1).csv.zip"

df = pd.read_csv(
    data_path,
    compression="zip",
    header=None,
    names=["userId", "productId", "rating", "timestamp"]
)

df["datetime"] = pd.to_datetime(df["timestamp"], unit="s")

print(df.shape)
df.head()

## 2. Preprocessing Decisions

Before defining the research questions, the raw ratings data must be converted into a representation suitable for pattern mining.

### Decision 1: Convert ratings into positive interactions
I define a positive interaction as `rating >= 4`.

**Why:**  
Phase 1 showed strong rating inflation. Since most ratings are positive, using exact rating values is less informative than modeling whether the user meaningfully engaged with or liked the product.

### Decision 2: Build user baskets
For frequent itemset mining, each user will be treated as one transaction basket containing positively rated products.

**Why:**  
Frequent itemset and association rule mining require transaction-style input.

### Decision 3: Filter extremely small baskets
Users with fewer than 2 positive products are not useful for co-occurrence mining, and users with fewer than 3 positive products are not useful for sequential pattern discovery.

**Why:**  
A basket of size 1 cannot contribute to item co-occurrence. Very short sequences also provide little sequential structure.

### Decision 4: Restrict the item universe for checkpoint experiments
Initial checkpoint experiments will use a filtered item universe, such as the top-N most frequent products among positive interactions.

**Why:**  
The full dataset contains hundreds of thousands of products. Running Apriori or sequential mining directly on the entire raw item space would be computationally expensive and likely produce extremely sparse, low-interpretability results. Filtering is necessary for feasibility at checkpoint stage.

In [ ]:
# ================================
# Positive interaction filtering
# ================================
df_pos = df[df["rating"] >= 4].copy()

print("Total interactions:", len(df))
print("Positive interactions (rating >= 4):", len(df_pos))
print("Positive interaction rate:", round(len(df_pos) / len(df), 4))

df_pos.head()

In [ ]:
# ================================
# Basket-size feasibility analysis
# ================================
basket_sizes = df_pos.groupby("userId")["productId"].nunique().sort_values(ascending=False)

print("Users with >=2 positive products:", (basket_sizes >= 2).sum())
print("Users with >=3 positive products:", (basket_sizes >= 3).sum())
print("Users with >=5 positive products:", (basket_sizes >= 5).sum())

basket_sizes.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
plt.figure(figsize=(8, 4))
basket_sizes.clip(upper=50).hist(bins=50)
plt.title("Positive Basket Size Distribution per User (capped at 50)")
plt.xlabel("Number of positively rated products")
plt.ylabel("User count")
plt.show()

In [ ]:
# ================================
# Product frequency feasibility analysis
# ================================
product_support = df_pos["productId"].value_counts()

print("Unique positive products:", product_support.shape[0])
print("Top 10 positive products:")
display(product_support.head(10))

print("\nProducts with at least 5 positive interactions:", (product_support >= 5).sum())
print("Products with at least 10 positive interactions:", (product_support >= 10).sum())
print("Products with at least 20 positive interactions:", (product_support >= 20).sum())

In [ ]:
top_ns = [100, 500, 1000, 2000, 5000]
total_pos = len(df_pos)

coverage_rows = []
for n in top_ns:
    top_items = set(product_support.head(n).index)
    covered = df_pos["productId"].isin(top_items).sum()
    coverage_rows.append({
        "top_n_products": n,
        "interaction_coverage": covered / total_pos
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

plt.figure(figsize=(7, 4))
plt.plot(coverage_df["top_n_products"], coverage_df["interaction_coverage"], marker="o")
plt.title("Coverage of Positive Interactions by Top-N Products")
plt.xlabel("Top-N products")
plt.ylabel("Fraction of positive interactions covered")
plt.show()

In [ ]:
# ================================
# Sequential feasibility analysis
# ================================
df_seq = df_pos.sort_values(["userId", "datetime"]).copy()

seq_lengths = df_seq.groupby("userId")["productId"].size()

print("Users with sequence length >= 3:", (seq_lengths >= 3).sum())
print("Users with sequence length >= 5:", (seq_lengths >= 5).sum())
print(seq_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
df_seq["gap_days"] = df_seq.groupby("userId")["datetime"].diff().dt.days

plt.figure(figsize=(8, 4))
df_seq["gap_days"].dropna().clip(upper=180).hist(bins=50)
plt.title("Time Gaps Between Positive Interactions (capped at 180 days)")
plt.xlabel("Gap in days")
plt.ylabel("Frequency")
plt.show()

df_seq["gap_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

## 3. Additional EDA Findings for Feasibility

The additional EDA supports several practical modeling decisions:

1. **Positive-interaction filtering is reasonable** because the dataset is dominated by high ratings, so a binary interaction view preserves the most meaningful signal.
2. **Minimum basket-size filtering is necessary** because many users interact with too few products to contribute useful co-occurrence patterns.
3. **Top-N item filtering is justified** because the product universe is extremely large, while a relatively small head of popular items covers a substantial fraction of all positive interactions.
4. **Sequential mining is feasible only after filtering** because many users have short histories, and long raw sequences over hundreds of thousands of products would be too sparse and expensive.
5. **Temporal gaps suggest ordering may matter**, which motivates including a sequential method in addition to unordered basket mining.

## 4. Research Questions

### RQ1. How do minimum support thresholds affect the quantity and quality of frequent itemsets and association rules in the Amazon Electronics dataset?
- This question focuses on sensitivity to support threshold.
- It examines whether lower support reveals more meaningful niche structure or mainly adds noise.

### RQ2. To what extent are discovered association rules dominated by product popularity, and can popularity-aware filtering improve rule diversity without severely hurting rule quality?
- This question focuses on popularity bias in rule mining.
- It tests whether naive rule mining mostly recovers obvious head-product relationships.

### RQ3. Do temporally ordered purchase sequences reveal patterns that are not captured by unordered frequent itemsets?
- This question compares unordered co-occurrence with ordered sequential structure.
- It evaluates whether a sequential mining method contributes genuinely new insight.

## 5. Why These Research Questions Are Non-Trivial

These questions are meaningful because the dataset has strong real-world complications:

- extreme sparsity,
- heavy popularity concentration,
- rating inflation,
- and temporal irregularity.

Because of these issues, the answers are not obvious in advance.

For example:
- Lowering support may uncover niche structure, or it may simply produce unstable patterns.
- High-confidence rules may look strong while actually reflecting only product popularity.
- Sequential ordering may add value, or it may provide little beyond static co-occurrence.

This makes the project more than a straightforward “run Apriori and report lift” exercise.

## 6. RQ-to-Method Mapping

| RQ | Task Type | Course or External | Planned Algorithm(s) | Why This Method Fits | Evaluation Criteria |
|---|---|---|---|---|---|
| RQ1 | Frequent itemset mining + association rule mining | Course | Apriori, FP-Growth | These are standard course methods for discovering co-occurrence structure under varying support thresholds. Using both helps compare tractability and consistency. | Number of itemsets, number of rules, support, confidence, lift, average rule length, coverage |
| RQ2 | Association rule analysis / bias analysis | Course | FP-Growth + association_rules + popularity-aware post-filtering | FP-Growth efficiently generates candidate rules; post-analysis can measure how much rules are driven by head products. | Confidence, lift, consequent popularity percentile, head-vs-tail ratio, rule diversity, catalog coverage |
| RQ3 | Sequential pattern mining | External | PrefixSpan | PrefixSpan is designed for ordered sequences and directly addresses temporal structure ignored by unordered baskets. | Sequential support, average sequence length, overlap with unordered itemsets, novelty, interpretability |

## 7. Motivation and Feasibility

### Motivation
The dataset exhibits long-tail behavior, popularity concentration, and temporal burstiness. These characteristics naturally motivate both unordered pattern mining and ordered sequence mining.

### Feasibility
The proposed methods are feasible with filtered data representations:
- frequent itemset mining can be run on positive user baskets after restricting the item universe,
- association rule analysis is straightforward once frequent itemsets are available,
- sequential pattern mining is feasible after filtering to users with sufficiently long histories and restricting the product space.

### Risks
There are several real risks:
- **computational cost:** Apriori can become expensive if the item universe is too large,
- **parameter sensitivity:** support thresholds may drastically change output volume,
- **popularity bias:** strong-looking rules may simply reflect head-item dominance,
- **sequence sparsity:** sequential mining may produce few interpretable patterns without filtering.

### Mitigation
To address these risks:
- use FP-Growth for larger filtered item sets,
- begin with small checkpoint runs before scaling,
- explicitly measure head-vs-tail dominance in rules,
- restrict sequence mining to filtered users and filtered items.

In [ ]:
# ================================
# Build filtered baskets for checkpoint experiments
# ================================
MIN_USER_ITEMS = 2
TOP_N_PRODUCTS = 1000

eligible_users = basket_sizes[basket_sizes >= MIN_USER_ITEMS].index
top_products = set(product_support.head(TOP_N_PRODUCTS).index)

df_basket = df_pos[
    df_pos["userId"].isin(eligible_users) &
    df_pos["productId"].isin(top_products)
].copy()

basket_df = (
    df_basket.groupby(["userId", "productId"])
    .size()
    .unstack(fill_value=0)
)

basket_binary = (basket_df > 0).astype(bool)

print("Filtered basket matrix shape:", basket_binary.shape)
print("Average basket size:", basket_binary.sum(axis=1).mean())
print("Median basket size:", basket_binary.sum(axis=1).median())

## 8. Checkpoint Experiment Design Choices

For the checkpoint experiments, I use:
- users with at least 2 positive products,
- products restricted to the top 1000 by positive interaction frequency.

These choices are not claimed to be globally optimal. They are used because they make the methods computationally feasible while preserving a meaningful fraction of the interaction signal.

This is appropriate for a checkpoint because the goal here is to verify that the proposed methods can run and produce interpretable results. Thresholds can be varied more systematically in the final phase.

In [ ]:
# ================================
# Initial Apriori sanity run
# ================================
apriori_itemsets = apriori(
    basket_binary,
    min_support=0.005,
    use_colnames=True
)

print("Apriori itemsets found:", len(apriori_itemsets))
display(apriori_itemsets.sort_values("support", ascending=False).head(10))

In [ ]:
# ================================
# Initial FP-Growth sanity run
# ================================
fp_itemsets = fpgrowth(
    basket_binary,
    min_support=0.005,
    use_colnames=True
)

print("FP-Growth itemsets found:", len(fp_itemsets))
display(fp_itemsets.sort_values("support", ascending=False).head(10))

In [ ]:
# ================================
# Initial association rule sanity run
# ================================
rules = association_rules(fp_itemsets, metric="confidence", min_threshold=0.2)

if len(rules) > 0:
    rules["antecedent_len"] = rules["antecedents"].apply(len)
    rules["consequent_len"] = rules["consequents"].apply(len)
    display(
        rules[["antecedents", "consequents", "support", "confidence", "lift", "antecedent_len", "consequent_len"]]
        .sort_values(["lift", "confidence"], ascending=False)
        .head(10)
    )
else:
    print("No rules found under current threshold.")

In [ ]:
# ================================
# Initial popularity bias analysis
# ================================
product_rank = product_support.rank(method="dense", ascending=False, pct=True)

def avg_popularity_percentile(itemset):
    items = list(itemset)
    return np.mean([product_rank.get(item, np.nan) for item in items])

if len(rules) > 0:
    rules["avg_antecedent_pop_pct"] = rules["antecedents"].apply(avg_popularity_percentile)
    rules["avg_consequent_pop_pct"] = rules["consequents"].apply(avg_popularity_percentile)

    display(
        rules[[
            "antecedents", "consequents", "confidence", "lift",
            "avg_antecedent_pop_pct", "avg_consequent_pop_pct"
        ]]
        .sort_values("confidence", ascending=False)
        .head(10)
    )

In [ ]:
# ================================
# Build filtered sequences
# ================================
MIN_SEQ_LEN = 3
TOP_N_SEQ_PRODUCTS = 500

top_seq_products = set(product_support.head(TOP_N_SEQ_PRODUCTS).index)

df_seq_filtered = df_pos[
    df_pos["productId"].isin(top_seq_products)
].sort_values(["userId", "datetime"]).copy()

user_sequences = (
    df_seq_filtered.groupby("userId")["productId"]
    .apply(list)
)

user_sequences = user_sequences[user_sequences.apply(len) >= MIN_SEQ_LEN]

sequences = user_sequences.tolist()

print("Number of sequences:", len(sequences))
print("Example sequence:", sequences[0][:10] if len(sequences) > 0 else "No sequences")

In [ ]:
# ================================
# Build filtered sequences
# ================================
MIN_SEQ_LEN = 3
TOP_N_SEQ_PRODUCTS = 500

top_seq_products = set(product_support.head(TOP_N_SEQ_PRODUCTS).index)

df_seq_filtered = df_pos[
    df_pos["productId"].isin(top_seq_products)
].sort_values(["userId", "datetime"]).copy()

user_sequences = (
    df_seq_filtered.groupby("userId")["productId"]
    .apply(list)
)

user_sequences = user_sequences[user_sequences.apply(len) >= MIN_SEQ_LEN]

sequences = user_sequences.tolist()

print("Number of sequences:", len(sequences))
print("Example sequence:", sequences[0][:10] if len(sequences) > 0 else "No sequences")

In [ ]:
# ================================
# Initial PrefixSpan sanity run
# ================================
ps = PrefixSpan(sequences)

# min support count, not fraction
min_count = max(5, int(0.005 * len(sequences)))

patterns = ps.frequent(min_count)

print("Number of sequential patterns found:", len(patterns))
print("Top 10 patterns by support:")
display(pd.DataFrame(patterns[:10], columns=["support_count", "sequence"]))

## 9. Initial Method Feasibility Results

The initial runs confirm that the proposed methods are workable on a filtered version of the dataset.

### Frequent itemset and rule mining
- Apriori and/or FP-Growth successfully produced itemsets on the filtered basket representation.
- Association rules were generated with interpretable support, confidence, and lift values.
- This confirms that RQ1 and RQ2 are computationally feasible.

### Popularity-bias analysis
- Initial inspection suggests that many strong rules may involve highly popular products.
- This supports the relevance of RQ2 rather than treating popularity bias as a purely theoretical concern.

### Sequential pattern mining
- PrefixSpan successfully produced ordered patterns on filtered user sequences.
- This confirms that RQ3 is technically feasible, provided that sequence construction and item filtering are handled carefully.

## 10. Final Method and Metric Plan

### For RQ1
- Run Apriori and FP-Growth across multiple minimum support thresholds.
- Compare:
  - number of frequent itemsets,
  - number of rules,
  - average support,
  - average confidence,
  - average lift,
  - catalog coverage.

### For RQ2
- Measure whether strong rules are concentrated among top-popularity items.
- Compare naive rule mining vs popularity-aware filtering.
- Evaluate:
  - head vs tail dominance,
  - rule diversity,
  - consequent coverage,
  - quality metrics such as lift and confidence.

### For RQ3
- Construct time-ordered sequences from positive user interactions.
- Run PrefixSpan on filtered sequences.
- Compare discovered sequential patterns against unordered frequent itemsets.
- Evaluate:
  - support,
  - novelty relative to unordered rules,
  - interpretability,
  - frequency of repeated ordered transitions.

### Baselines
- High-support-only rule mining baseline for RQ1 and RQ2.
- Unordered basket mining baseline for RQ3.

### Expected contribution
The project aims to show not only what patterns exist, but also which patterns are trivial, which are biased by popularity, and which truly require temporal ordering to be detected.

# Collaboration Declaration

## (1) Collaborators
I completed this work independently.

## (2) Web Sources
- UCSD Amazon product dataset documentation
- mlxtend documentation
- PrefixSpan package documentation
- Any additional sources used for implementation details will be cited in the final notebook

## (3) AI Tools
- ChatGPT was used for idea refinement, notebook organization, and explanation polishing

## (4) Citations for Papers
- McAuley, J., Pandey, R., & Leskovec, J. (2015). Inferring networks of substitutable and complementary products.
- Any additional papers used in the final phase will be cited explicitly.